# Problem Statement

## **Business Context**

"Visit with Us," a leading travel company, is revolutionizing the tourism industry by leveraging data-driven strategies to optimize operations and customer engagement. While introducing a new package offering, such as the Wellness Tourism Package, the company faces challenges in targeting the right customers efficiently. The manual approach to identifying potential customers is inconsistent, time-consuming, and prone to errors, leading to missed opportunities and suboptimal campaign performance.

To address these issues, the company aims to implement a scalable and automated system that integrates customer data, predicts potential buyers, and enhances decision-making for marketing strategies. By utilizing an MLOps pipeline, the company seeks to achieve seamless integration of data preprocessing, model development, deployment, and CI/CD practices for continuous improvement. This system will ensure efficient targeting of customers, timely updates to the predictive model, and adaptation to evolving customer behaviors, ultimately driving growth and customer satisfaction.


## **Objective**

As an MLOps Engineer at "Visit with Us," your responsibility is to design and deploy an MLOps pipeline on GitHub to automate the end-to-end workflow for predicting customer purchases. The primary objective is to build a model that predicts whether a customer will purchase the newly introduced Wellness Tourism Package before contacting them. The pipeline will include data cleaning, preprocessing, transformation, model building, training, evaluation, and deployment, ensuring consistent performance and scalability. By leveraging GitHub Actions for CI/CD integration, the system will enable automated updates, streamline model deployment, and improve operational efficiency. This robust predictive solution will empower policymakers to make data-driven decisions, enhance marketing strategies, and effectively target potential customers, thereby driving customer acquisition and business growth.

## **Data Description**

The dataset contains customer and interaction data that serve as key attributes for predicting the likelihood of purchasing the Wellness Tourism Package. The detailed attributes are:

**Customer Details**
- **CustomerID:** Unique identifier for each customer.
- **ProdTaken:** Target variable indicating whether the customer has purchased a package (0: No, 1: Yes).
- **Age:** Age of the customer.
- **TypeofContact:** The method by which the customer was contacted (Company Invited or Self Inquiry).
- **CityTier:** The city category based on development, population, and living standards (Tier 1 > Tier 2 > Tier 3).
- **Occupation:** Customer's occupation (e.g., Salaried, Freelancer).
- **Gender:** Gender of the customer (Male, Female).
- **NumberOfPersonVisiting:** Total number of people accompanying the customer on the trip.
- **PreferredPropertyStar:** Preferred hotel rating by the customer.
- **MaritalStatus:** Marital status of the customer (Single, Married, Divorced).
- **NumberOfTrips:** Average number of trips the customer takes annually.
- **Passport:** Whether the customer holds a valid passport (0: No, 1: Yes).
- **OwnCar:** Whether the customer owns a car (0: No, 1: Yes).
- **NumberOfChildrenVisiting:** Number of children below age 5 accompanying the customer.
- **Designation:** Customer's designation in their current organization.
- **MonthlyIncome:** Gross monthly income of the customer.

**Customer Interaction Data**
- **PitchSatisfactionScore:** Score indicating the customer's satisfaction with the sales pitch.
- **ProductPitched:** The type of product pitched to the customer.
- **NumberOfFollowups:** Total number of follow-ups by the salesperson after the sales pitch.-
- **DurationOfPitch:** Duration of the sales pitch delivered to the customer.


## Data Registration

In [6]:
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from huggingface_hub import login, HfApi
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Load HF token from .env file
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

# Login to Hugging Face
login(token=HF_TOKEN)
print("Logged in to Hugging Face successfully")

Logged in to Hugging Face successfully


In [7]:
import os

os.makedirs("tourism_project/data", exist_ok=True)
os.makedirs("tourism_project/model_building", exist_ok=True)
os.makedirs("tourism_project/deployment", exist_ok=True)

print("Folders created successfully")

Folders created successfully


In [8]:
api = HfApi()

# Upload raw CSV to HF dataset repo
api.upload_file(
    path_or_fileobj="data/tourism.csv",
    path_in_repo="tourism.csv",
    repo_id="SANGU19/tourism-dataset",
    repo_type="dataset"
)

print("Raw dataset uploaded to SANGU19/tourism-dataset successfully")

Raw dataset uploaded to SANGU19/tourism-dataset successfully


## Data Preparation

In [10]:
df = pd.read_csv("data/tourism.csv")

print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum())
print("\nTarget distribution:")
print(df["ProdTaken"].value_counts())
print("\nSample rows:")
df.head()

Shape: (4128, 21)

Column names:
['Unnamed: 0', 'CustomerID', 'ProdTaken', 'Age', 'TypeofContact', 'CityTier', 'DurationOfPitch', 'Occupation', 'Gender', 'NumberOfPersonVisiting', 'NumberOfFollowups', 'ProductPitched', 'PreferredPropertyStar', 'MaritalStatus', 'NumberOfTrips', 'Passport', 'PitchSatisfactionScore', 'OwnCar', 'NumberOfChildrenVisiting', 'Designation', 'MonthlyIncome']

Missing values:
Unnamed: 0                  0
CustomerID                  0
ProdTaken                   0
Age                         0
TypeofContact               0
CityTier                    0
DurationOfPitch             0
Occupation                  0
Gender                      0
NumberOfPersonVisiting      0
NumberOfFollowups           0
ProductPitched              0
PreferredPropertyStar       0
MaritalStatus               0
NumberOfTrips               0
Passport                    0
PitchSatisfactionScore      0
OwnCar                      0
NumberOfChildrenVisiting    0
Designation                

,Unnamed: 0,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,...,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,...,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0
1,1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,...,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,...,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0
3,3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,...,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,5,200005,0,32.0,Company Invited,1,8.0,Salaried,Male,3,...,Basic,3.0,Single,1.0,0,5,1,1.0,Executive,18068.0


In [11]:
# Drop CustomerID - identifier, not a feature
df.drop(columns=["CustomerID"], inplace=True)

# Fill missing numerical columns with median
num_cols = df.select_dtypes(include=["float64", "int64"]).columns
for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Fill missing categorical columns with mode
cat_cols = df.select_dtypes(include=["object"]).columns
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

print("Missing values after cleaning:")
print(df.isnull().sum())
print("\nShape after cleaning:", df.shape)

Missing values after cleaning:
Unnamed: 0                  0
ProdTaken                   0
Age                         0
TypeofContact               0
CityTier                    0
DurationOfPitch             0
Occupation                  0
Gender                      0
NumberOfPersonVisiting      0
NumberOfFollowups           0
ProductPitched              0
PreferredPropertyStar       0
MaritalStatus               0
NumberOfTrips               0
Passport                    0
PitchSatisfactionScore      0
OwnCar                      0
NumberOfChildrenVisiting    0
Designation                 0
MonthlyIncome               0
dtype: int64

Shape after cleaning: (4128, 20)


C:\Users\DELL\AppData\Local\Temp\ipykernel_11836\4045566999.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
C:\Users\DELL\AppData\Local\Temp\ipykernel_11836\4045566999.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example

In [12]:
le = LabelEncoder()

for col in df.select_dtypes(include=["object"]).columns:
    df[col] = le.fit_transform(df[col])
    print(f"Encoded: {col}")

print("\nAll categorical columns encoded successfully")
print("\nFinal dataframe shape:", df.shape)
df.head()

Encoded: TypeofContact
Encoded: Occupation
Encoded: Gender
Encoded: ProductPitched
Encoded: MaritalStatus
Encoded: Designation

All categorical columns encoded successfully

Final dataframe shape: (4128, 20)


,Unnamed: 0,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,0,1,41.0,1,3,6.0,2,1,3,3.0,1,3.0,2,1.0,1,2,1,0.0,2,20993.0
1,1,0,49.0,0,1,14.0,2,2,3,4.0,1,4.0,0,2.0,0,3,1,2.0,2,20130.0
2,2,1,37.0,1,1,8.0,0,2,3,4.0,0,3.0,2,7.0,1,3,0,0.0,1,17090.0
3,3,0,33.0,0,1,9.0,2,1,2,3.0,0,3.0,0,2.0,1,5,1,1.0,1,17909.0
4,5,0,32.0,0,1,8.0,2,2,3,3.0,0,3.0,2,1.0,0,5,1,1.0,1,18068.0


In [13]:
X = df.drop(columns=["ProdTaken"])
y = df["ProdTaken"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Combine features and target back for saving
train_df = X_train.copy()
train_df["ProdTaken"] = y_train.values

test_df = X_test.copy()
test_df["ProdTaken"] = y_test.values

# Save locally
train_df.to_csv("tourism_project/data/train.csv", index=False)
test_df.to_csv("tourism_project/data/test.csv", index=False)

print(f"Train size: {train_df.shape}")
print(f"Test size:  {test_df.shape}")
print("\nFiles saved to tourism_project/data/")

Train size: (3302, 20)
Test size:  (826, 20)

Files saved to tourism_project/data/


In [14]:
# Upload train split
api.upload_file(
    path_or_fileobj="tourism_project/data/train.csv",
    path_in_repo="train.csv",
    repo_id="SANGU19/tourism-dataset",
    repo_type="dataset"
)

# Upload test split
api.upload_file(
    path_or_fileobj="tourism_project/data/test.csv",
    path_in_repo="test.csv",
    repo_id="SANGU19/tourism-dataset",
    repo_type="dataset"
)

print("train.csv and test.csv uploaded to SANGU19/tourism-dataset successfully")

train.csv and test.csv uploaded to SANGU19/tourism-dataset successfully


In [17]:
os.makedirs("tourism_project/data", exist_ok=True)

## Model Training and Registration with Experimentation Tracking

In [15]:
!pip install mlflow xgboost -q
print("MLflow and XGBoost installed")

MLflow and XGBoost installed


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

In [18]:
pip install mlflow

  Using cached mlflow-3.11.1-py3-none-any.whl.metadata (49 kB)
  Using cached mlflow_skinny-3.11.1-py3-none-any.whl.metadata (49 kB)
  Using cached mlflow_tracing-3.11.1-py3-none-any.whl.metadata (19 kB)
  Using cached alembic-1.18.4-py3-none-any.whl.metadata (7.2 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached huey-2.6.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached skops-0.14.0-py3-none-any.whl.metadata (4.4 kB)
  Using cached waitress-3.0.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached cachetools-7.0.6-py3-none-any.whl.metadata (5.9 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached databricks_sdk-0.106.0-py3-none-any.whl.metadata (43 kB)
  Using cached fastapi-0.136.1-py3-none-any.whl.metadata (28 kB)
  Using cached gitpython-3.1.49-py3-none-any.whl.metadata (14 kB)
  Using cached importlib_metadata-8.9.0-py3-none-any.whl.metadata (4.5 k


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score
)
from huggingface_hub import HfApi
import pickle
import os

api = HfApi()
print("Imports successful")

Imports successful


In [20]:
import pandas as pd

train_df = pd.read_csv("hf://datasets/SANGU19/tourism-dataset/train.csv")
test_df  = pd.read_csv("hf://datasets/SANGU19/tourism-dataset/test.csv")

X_train = train_df.drop(columns=["ProdTaken"])
y_train = train_df["ProdTaken"]

X_test = test_df.drop(columns=["ProdTaken"])
y_test = test_df["ProdTaken"]

print("Train shape:", X_train.shape)
print("Test shape: ", X_test.shape)

Train shape: (3302, 19)
Test shape:  (826, 19)


In [21]:
# MLflow will store logs locally in mlruns/ folder
mlflow.set_experiment("tourism_purchase_prediction")

print("MLflow experiment set: tourism_purchase_prediction")
print("Logs will be stored in ./mlruns/")

2026/05/02 00:42:16 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/02 00:42:16 INFO mlflow.store.db.utils: Updating database tables
2026/05/02 00:42:33 INFO mlflow.tracking.fluent: Experiment with name 'tourism_purchase_prediction' does not exist. Creating a new experiment.


MLflow experiment set: tourism_purchase_prediction
Logs will be stored in ./mlruns/


In [22]:
rf_params = {
    "n_estimators": [100, 200],
    "max_depth": [4, 6, 8],
    "min_samples_split": [2, 5]
}

with mlflow.start_run(run_name="RandomForest_GridSearch"):

    rf = RandomForestClassifier(random_state=42)

    rf_grid = GridSearchCV(
        rf, rf_params,
        cv=5,
        scoring="f1",
        n_jobs=-1
    )
    rf_grid.fit(X_train, y_train)

    best_rf = rf_grid.best_estimator_
    y_pred_rf = best_rf.predict(X_test)

    # Metrics
    acc_rf  = accuracy_score(y_test, y_pred_rf)
    prec_rf = precision_score(y_test, y_pred_rf)
    rec_rf  = recall_score(y_test, y_pred_rf)
    f1_rf   = f1_score(y_test, y_pred_rf)
    auc_rf  = roc_auc_score(y_test, y_pred_rf)

    # Log params and metrics to MLflow
    mlflow.log_params(rf_grid.best_params_)
    mlflow.log_metric("accuracy", acc_rf)
    mlflow.log_metric("precision", prec_rf)
    mlflow.log_metric("recall", rec_rf)
    mlflow.log_metric("f1_score", f1_rf)
    mlflow.log_metric("roc_auc", auc_rf)

    mlflow.sklearn.log_model(best_rf, "random_forest_model")

    print("=== Random Forest Results ===")
    print(f"Best Params : {rf_grid.best_params_}")
    print(f"Accuracy    : {acc_rf:.4f}")
    print(f"Precision   : {prec_rf:.4f}")
    print(f"Recall      : {rec_rf:.4f}")
    print(f"F1 Score    : {f1_rf:.4f}")
    print(f"ROC AUC     : {auc_rf:.4f}")

2026/05/02 00:43:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 00:43:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


=== Random Forest Results ===
Best Params : {'max_depth': 8, 'min_samples_split': 2, 'n_estimators': 100}
Accuracy    : 0.8680
Precision   : 0.8472
Recall      : 0.3836
F1 Score    : 0.5281
ROC AUC     : 0.6836


In [23]:
gb_params = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "max_depth": [3, 5]
}

with mlflow.start_run(run_name="GradientBoosting_GridSearch"):

    gb = GradientBoostingClassifier(random_state=42)

    gb_grid = GridSearchCV(
        gb, gb_params,
        cv=5,
        scoring="f1",
        n_jobs=-1
    )
    gb_grid.fit(X_train, y_train)

    best_gb = gb_grid.best_estimator_
    y_pred_gb = best_gb.predict(X_test)

    # Metrics
    acc_gb  = accuracy_score(y_test, y_pred_gb)
    prec_gb = precision_score(y_test, y_pred_gb)
    rec_gb  = recall_score(y_test, y_pred_gb)
    f1_gb   = f1_score(y_test, y_pred_gb)
    auc_gb  = roc_auc_score(y_test, y_pred_gb)

    # Log params and metrics to MLflow
    mlflow.log_params(gb_grid.best_params_)
    mlflow.log_metric("accuracy", acc_gb)
    mlflow.log_metric("precision", prec_gb)
    mlflow.log_metric("recall", rec_gb)
    mlflow.log_metric("f1_score", f1_gb)
    mlflow.log_metric("roc_auc", auc_gb)

    mlflow.sklearn.log_model(best_gb, "gradient_boosting_model")

    print("=== Gradient Boosting Results ===")
    print(f"Best Params : {gb_grid.best_params_}")
    print(f"Accuracy    : {acc_gb:.4f}")
    print(f"Precision   : {prec_gb:.4f}")
    print(f"Recall      : {rec_gb:.4f}")
    print(f"F1 Score    : {f1_gb:.4f}")
    print(f"ROC AUC     : {auc_gb:.4f}")

2026/05/02 00:44:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 00:44:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


=== Gradient Boosting Results ===
Best Params : {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200}
Accuracy    : 0.9310
Precision   : 0.9322
Recall      : 0.6918
F1 Score    : 0.7942
ROC AUC     : 0.8399


In [24]:
# Compare both models on F1 score
results = {
    "RandomForest": {"model": best_rf, "f1": f1_rf},
    "GradientBoosting": {"model": best_gb, "f1": f1_gb}
}

best_name = max(results, key=lambda x: results[x]["f1"])
best_model = results[best_name]["model"]

print(f"Best model: {best_name}")
print(f"F1 Score  : {results[best_name]['f1']:.4f}")

# Save model locally
os.makedirs("tourism_project/model_building", exist_ok=True)
model_path = "tourism_project/model_building/best_model.pkl"

with open(model_path, "wb") as f:
    pickle.dump(best_model, f)

print(f"\nModel saved locally at: {model_path}")

Best model: GradientBoosting
F1 Score  : 0.7942

Model saved locally at: tourism_project/model_building/best_model.pkl


In [25]:
api.upload_file(
    path_or_fileobj="tourism_project/model_building/best_model.pkl",
    path_in_repo="best_model.pkl",
    repo_id="SANGU19/tourism-model",
    repo_type="model"
)

print(f"Best model ({best_name}) uploaded to SANGU19/tourism-model successfully")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Best model (GradientBoosting) uploaded to SANGU19/tourism-model successfully


# Deployment

## Dockerfile

In [27]:
pip install streamlit



[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip



   ---------------------------------------- 0.0/9.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.2 MB ? eta -:--:--
   - ----------------------------

## Dependency Handling

Please ensure that the dependency handling file is named `requirements.txt`.

# Hosting

# MLOps Pipeline with Github Actions Workflow

**Note:**

1. Before running the file below, make sure to add the HF_TOKEN to your GitHub secrets to enable authentication between GitHub and Hugging Face.
2. The below code is for a sample YAML file that can be updated as required to meet the requirements of this project.

```
name: Tourism Project Pipeline

on:
  push:
    branches:
      - main  # Automatically triggers on push to the main branch

jobs:

  register-dataset:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: <add_code_here>
      - name: Upload Dataset to Hugging Face Hub
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: <add_code_here>

  data-prep:
    needs: register-dataset
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: <add_code_here>
      - name: Run Data Preparation
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: <add_code_here>


  model-traning:
    needs: data-prep
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: <add_code_here>
      - name: Start MLflow Server
        run: |
          nohup mlflow ui --host 0.0.0.0 --port 5000 &  # Run MLflow UI in the background
          sleep 5  # Wait for a moment to let the server starts
      - name: Model Building
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: <add_code_here>


  deploy-hosting:
    runs-on: ubuntu-latest
    needs: [model-traning,data-prep,register-dataset]
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: <add_code_here>
      - name: Push files to Frontend Hugging Face Space
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: <add_code_here>

```

**Note:** To use this YAML file for our use case, we need to

1. Go to the GitHub repository for the project
2. Create a folder named ***.github/workflows/***
3. In the above folder, create a file named ***pipeline.yml***
4. Copy and paste the above content for the YAML file into the ***pipeline.yml*** file

## Requirements file for the Github Actions Workflow

## Github Authentication and Push Files

* Before moving forward, we need to generate a secret token to push files directly from Colab to the GitHub repository.
* Please follow the below instructions to create the GitHub token:
    - Open your GitHub profile.
    - Click on ***Settings***.
    - Go to ***Developer Settings***.
    - Expand the ***Personal access tokens*** section and select ***Tokens (classic)***.
    - Click ***Generate new token***, then choose ***Generate new token (classic)***.
    - Add a note and select all required scopes.
    - Click ***Generate token***.
    - Copy the generated token and store it safely in a notepad.

In [ ]:
# Install Git
!apt-get install git

# Set your Git identity (replace with your details)
!git config --global user.email "<-------GitHub Email Address------->"
!git config --global user.name "<--------GitHub UserName--------->"

# Clone your GitHub repository
!git clone https://github.com/<--------GitHub UserName--------->/<--------GitHub Reponame--------->.git

# Move your folder to the repository directory
!mv /content/tourism_project/ /content/<--------GitHub Reponame--------->

In [ ]:
# Change directory to the cloned repository
%cd <--------GitHub Reponame--------->/

# Add the new folder to Git
!git add .

# Commit the changes
!git commit -m "first commit"

# Push to GitHub (you'll need your GitHub credentials; use a personal access token if 2FA enabled)
!git push https://<--------GitHub UserName--------->:<--------GitHub Token--------->@github.com/<--------GitHub UserName--------->/<--------GitHub Reponame--------->.git

# Output Evaluation

- GitHub (link to repository, screenshot of folder structure and executed workflow)

- Streamlit on Hugging Face (link to HF space, screenshot of Streamlit app)

<font size=6 color="navyblue">Power Ahead!</font>
___